In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import numpy as np
import torch
import pytorch_lightning as pl

import data as my_data
from models.gated_recurrent_unit import GatedRecurrentUnit
from lightning.lightning import LightningRNNOneHot
from torch.utils.data import DataLoader


SEED = 2334
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
np.random.seed(SEED)


c:\DEV\RNN_test\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
MODULE = LightningRNNOneHot

In [4]:
encoder, total_samples = my_data.load('../data/*.txt')

train_samples, val_samples = np.split(total_samples,
                                        [int(.9 * len(total_samples))])
print("Total samples:{} = train:{}, valid:{}".format(
    len(total_samples), len(train_samples), len(val_samples)))
del total_samples

n_categories = len(encoder.all_categories)
input_size = len(encoder.all_letters)
output_size = len(encoder.all_letters)

train_dl = DataLoader(my_data.CityNamesOneHot(encoder, train_samples, device="cuda"),
                          shuffle=False,
                          num_workers=6,
                          batch_size=8,
                          collate_fn=my_data.pad_collate)
val_dl = DataLoader(my_data.CityNamesOneHot(encoder, val_samples, device="cuda"),
                          shuffle=False,
                          batch_size=1,
                          collate_fn=my_data.pad_collate)

# categories: 2, ['ru', 'us'] samples:
ru: 1117
us: 1960
Total chars: 121
Total samples:3077 = train:2769, valid:308


In [24]:
# train_samples = my_data.train_samples(encoder)
# val_samples = my_data.val_samples(encoder)
model = MODULE(GatedRecurrentUnit(input_size, output_size))
trainer = pl.Trainer(max_epochs=10)
trainer.fit(model, 
            train_dataloaders=train_dl, 
            val_dataloaders=val_dl)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name  | Type               | Params
---------------------------------------------
0 | model | GatedRecurrentUnit | 166 K 
---------------------------------------------
166 K     Trainable params
0         Non-trainable params
166 K     Total params
0.665     Total estimated model params size (MB)


Epoch 0:   0%|          | 0/347 [00:00<?, ?it/s]                            torch.Size([8, 9, 166]) torch.Size([8, 9])


RuntimeError: For unbatched 2-D input, hx should also be 2-D but got 1-D tensor